# Fine-tune BART on CNN/DailyMail

This notebook fine-tunes `facebook/bart-large-cnn` for one epoch. GPU is strongly recommended; CPU runs will be very slow.

In [ ]:
%pip install datasets==2.19.0 evaluate transformers==4.57.1 torch rouge-score sentencepiece

In [ ]:
from datasets import load_dataset

dataset = load_dataset("cnn_dailymail", "3.0.0")
dataset

In [ ]:
from transformers import AutoTokenizer

model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_batch(batch):
    inputs = tokenizer(batch["article"], max_length=1024, truncation=True)
    labels = tokenizer(batch["highlights"], max_length=128, truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized = dataset.map(tokenize_batch, batched=True, remove_columns=dataset["train"].column_names)
tokenized

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./models/bart-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_steps=500,
    save_total_limit=1,
    evaluation_strategy="epoch",
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"].select(range(1000)),
    tokenizer=tokenizer,
    data_collator=collator,
)
trainer.train()

In [ ]:
import evaluate

rouge = evaluate.load("rouge")
sample = tokenized["test"].select(range(100))
predictions = trainer.predict(sample)
decoded_preds = tokenizer.batch_decode(predictions.predictions, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(predictions.label_ids, skip_special_tokens=True)
rouge.compute(predictions=decoded_preds, references=decoded_labels)

In [ ]:
trainer.save_model("./models/bart-finetuned")
tokenizer.save_pretrained("./models/bart-finetuned")

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {"model": "baseline", "rouge1": None, "rouge2": None, "rougeL": None},
    {"model": "finetuned", "rouge1": "fill from eval", "rouge2": "fill from eval", "rougeL": "fill from eval"},
])
comparison